In [ ]:
# =============================================================================
# 🧹 NETTOYAGE SYSTÈME (SSH) - EXÉCUTER EN PREMIER!
# =============================================================================

import os, gc, shutil, glob

def quick_cleanup():
    """Nettoyage rapide avant exécution."""
    print("🧹 NETTOYAGE...")
    gc.collect()
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("✅ Cache CUDA vidé")
    except: pass
    
    for pattern in ['**/__pycache__', '**/.ipynb_checkpoints']:
        for p in glob.glob(pattern, recursive=True):
            try: shutil.rmtree(p)
            except: pass
    
    total, used, free = shutil.disk_usage('/')
    print(f"💾 Espace: {free/1e9:.1f} GB libre")
    print("✅ Prêt!")

quick_cleanup()

In [1]:
# Imports & configuration
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
# Paths (adapt if needed)
DATA_DIR = os.path.join('..','data') if os.path.exists(os.path.join('..','data')) else './data'
EMB_DIR = os.path.join(DATA_DIR, 'embeddings')
FEAT_DIR = os.path.join(DATA_DIR, 'features')
Y_PATH = os.path.join(DATA_DIR, 'y_train.npy')

train_text_path = os.path.join(EMB_DIR, 'X_train_text_only_embeddings.npy')
train_desc_path = os.path.join(EMB_DIR, 'X_train_desc_only_embeddings.npy')
kaggle_text_path = os.path.join(EMB_DIR, 'X_kaggle_text_only_embeddings.npy')
kaggle_desc_path = os.path.join(EMB_DIR, 'X_kaggle_desc_only_embeddings.npy')

train_feat_path = os.path.join(FEAT_DIR, 'X_train_features.npy')
kaggle_feat_path = os.path.join(FEAT_DIR, 'X_kaggle_features.npy')

for p in [train_text_path, train_desc_path, kaggle_text_path, kaggle_desc_path, train_feat_path, kaggle_feat_path, Y_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing required file: {p}')

# Load arrays
X_train_t = np.load(train_text_path).astype(np.float32)
X_train_u = np.load(train_desc_path).astype(np.float32)
X_kaggle_t = np.load(kaggle_text_path).astype(np.float32)
X_kaggle_u = np.load(kaggle_desc_path).astype(np.float32)

X_train_meta = np.load(train_feat_path)
X_kaggle_meta = np.load(kaggle_feat_path)

y = np.load(Y_PATH)

# --- Quick cleaning to avoid NaN/inf and ensure valid integer labels ---
from sklearn.preprocessing import LabelEncoder

# Encode labels to 0..C-1 (handles strings or unexpected label values)
le = LabelEncoder()
y = le.fit_transform(y)

# Clean function to replace NaN/inf and cast to float32
def clean(a):
    a = np.asarray(a)
    a = np.nan_to_num(a, nan=0.0, posinf=1e6, neginf=-1e6)
    return a.astype(np.float32)

X_train_t = clean(X_train_t)
X_train_u = clean(X_train_u)
X_train_meta = clean(X_train_meta)
X_kaggle_t = clean(X_kaggle_t)
X_kaggle_u = clean(X_kaggle_u)
X_kaggle_meta = clean(X_kaggle_meta)

print('Shapes:')
print('train tweet emb:', X_train_t.shape)
print('train user emb :', X_train_u.shape)
print('train meta      :', X_train_meta.shape)
print('labels          :', y.shape)
print('kaggle tweet emb:', X_kaggle_t.shape)
print('kaggle user emb :', X_kaggle_u.shape)
print('kaggle meta      :', X_kaggle_meta.shape)

n = X_train_t.shape[0]
assert X_train_u.shape[0] == n and X_train_meta.shape[0] == n and y.shape[0] == n, 'Train row mismatch'
assert X_kaggle_t.shape[0] == X_kaggle_u.shape[0] == X_kaggle_meta.shape[0], 'Kaggle row mismatch'

meta_dim = X_train_meta.shape[1]
tweet_dim = X_train_t.shape[1]
user_dim = X_train_u.shape[1]
n_classes = len(le.classes_)
print(f'meta_dim={meta_dim}, tweet_dim={tweet_dim}, user_dim={user_dim}, n_classes={n_classes}')

Shapes:
train tweet emb: (154914, 768)
train user emb : (154914, 768)
train meta      : (154914, 152)
labels          : (154914,)
kaggle tweet emb: (103380, 768)
kaggle user emb : (103380, 768)
kaggle meta      : (103380, 152)
meta_dim=152, tweet_dim=768, user_dim=768, n_classes=2


In [3]:
# Dataset and DataLoaders
class InfluencerDataset(Dataset):
    def __init__(self, tweet_emb, user_emb, meta, labels=None):
        self.tweet = torch.from_numpy(tweet_emb).float()
        self.user = torch.from_numpy(user_emb).float()
        self.meta = torch.from_numpy(meta).float()
        self.labels = None if labels is None else torch.from_numpy(labels).long()
    def __len__(self):
        return self.tweet.shape[0]
    def __getitem__(self, idx):
        if self.labels is None:
            return self.tweet[idx], self.user[idx], self.meta[idx]
        return self.tweet[idx], self.user[idx], self.meta[idx], self.labels[idx]

# Split
X_t_train, X_t_val, X_u_train, X_u_val, X_m_train, X_m_val, y_train, y_val = train_test_split(
    X_train_t, X_train_u, X_train_meta, y, test_size=0.15, random_state=42, stratify=y)

train_dataset = InfluencerDataset(X_t_train, X_u_train, X_m_train, y_train)
val_dataset = InfluencerDataset(X_t_val, X_u_val, X_m_val, y_val)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
print('Train samples:', len(train_dataset), 'Val samples:', len(val_dataset))

Train samples: 131676 Val samples: 23238


In [4]:
# Model per spec
class InfluencerModel(nn.Module):
    def __init__(self, tweet_dim, user_dim, meta_dim, n_classes):
        super().__init__()
        self.tweet_branch = nn.Sequential(nn.Linear(tweet_dim, 256), nn.ReLU(inplace=True), nn.Dropout(0.2))
        self.user_branch = nn.Sequential(nn.Linear(user_dim, 256), nn.ReLU(inplace=True), nn.Dropout(0.2))
        self.meta_branch = nn.Sequential(nn.Linear(meta_dim, 32), nn.ReLU(inplace=True))
        total = 256 + 256 + 32
        self.head = nn.Sequential(nn.Linear(total, 128), nn.ReLU(inplace=True), nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Linear(64, n_classes))
    def forward(self, tweet, user, meta):
        t = self.tweet_branch(tweet)
        u = self.user_branch(user)
        m = self.meta_branch(meta)
        x = torch.cat([t, u, m], dim=1)
        return self.head(x)

model = InfluencerModel(tweet_dim, user_dim, meta_dim, n_classes).to(device)
print(model)

InfluencerModel(
  (tweet_branch): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.2, inplace=False)
  )
  (user_branch): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.2, inplace=False)
  )
  (meta_branch): Sequential(
    (0): Linear(in_features=152, out_features=32, bias=True)
    (1): ReLU(inplace=True)
  )
  (head): Sequential(
    (0): Linear(in_features=544, out_features=128, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [10]:
# Training utilities
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses, preds_all, labels_all = [], [], []
    for batch in loader:
        tweet, user, meta, labels = batch
        tweet, user, meta, labels = tweet.to(device), user.to(device), meta.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(tweet, user, meta)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average='macro') if len(np.unique(labels_all))>1 else 0.0
    return np.mean(losses), acc, f1

def eval_model(model, loader, criterion, device):
    model.eval()
    losses, preds_all, labels_all = [], [], []
    with torch.no_grad():
        for batch in loader:
            tweet, user, meta, labels = batch
            tweet, user, meta, labels = tweet.to(device), user.to(device), meta.to(device), labels.to(device)
            logits = model(tweet, user, meta)
            loss = criterion(logits, labels)
            losses.append(loss.item())
            preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            labels_all.extend(labels.cpu().numpy().tolist())
    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average='macro') if len(np.unique(labels_all))>1 else 0.0
    return np.mean(losses), acc, f1

epochs = 15
best_val_f1 = -1.0
os.makedirs('../models', exist_ok=True)
checkpoint_path = '../models/influencer_model.pt'

In [11]:
# Training loop (runs epochs)
for epoch in range(1, epochs+1):
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_f1 = eval_model(model, val_loader, criterion, device)
    print(f'Epoch {epoch}/{epochs}  train_loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f}  |  val_loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}')
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'model_state_dict': model.state_dict(), 'meta_dim': meta_dim, 'tweet_dim': tweet_dim, 'user_dim': user_dim, 'n_classes': n_classes}, checkpoint_path)
        print('Checkpoint saved to', checkpoint_path)

print('Training complete. Best val f1:', best_val_f1)

Epoch 1/15  train_loss=0.2107 acc=0.9160 f1=0.9153  |  val_loss=0.2859 acc=0.8894 f1=0.8880
Checkpoint saved to ../models/influencer_model.pt
Epoch 2/15  train_loss=0.2067 acc=0.9168 f1=0.9162  |  val_loss=0.2501 acc=0.8993 f1=0.8984
Checkpoint saved to ../models/influencer_model.pt
Epoch 2/15  train_loss=0.2067 acc=0.9168 f1=0.9162  |  val_loss=0.2501 acc=0.8993 f1=0.8984
Checkpoint saved to ../models/influencer_model.pt
Epoch 3/15  train_loss=0.2035 acc=0.9184 f1=0.9178  |  val_loss=0.2751 acc=0.8881 f1=0.8879
Epoch 3/15  train_loss=0.2035 acc=0.9184 f1=0.9178  |  val_loss=0.2751 acc=0.8881 f1=0.8879
Epoch 4/15  train_loss=0.2037 acc=0.9185 f1=0.9179  |  val_loss=0.2476 acc=0.9028 f1=0.9018
Checkpoint saved to ../models/influencer_model.pt
Epoch 4/15  train_loss=0.2037 acc=0.9185 f1=0.9179  |  val_loss=0.2476 acc=0.9028 f1=0.9018
Checkpoint saved to ../models/influencer_model.pt
Epoch 5/15  train_loss=0.2048 acc=0.9171 f1=0.9164  |  val_loss=0.2468 acc=0.9036 f1=0.9029
Checkpoint sav

In [12]:
# Inference on Kaggle set
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
model.eval()

kaggle_dataset = InfluencerDataset(X_kaggle_t, X_kaggle_u, X_kaggle_meta, labels=None)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
preds = []
with torch.no_grad():
    for batch in kaggle_loader:
        tweet, user, meta = batch
        tweet, user, meta = tweet.to(device), user.to(device), meta.to(device)
        logits = model(tweet, user, meta)
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())

# Build submission
kaggle_df = pd.read_json(os.path.join(DATA_DIR, 'kaggle_test.jsonl'), lines=True)
kaggle_df = pd.json_normalize(kaggle_df.to_dict(orient='records'))
ids = kaggle_df['challenge_id'].astype(int).values if 'challenge_id' in kaggle_df.columns else np.arange(len(preds))
submission = pd.DataFrame({'ID': ids, 'label': preds})
os.makedirs('../submission', exist_ok=True)
submission_path = '../submission/submission_NN.csv'
submission.to_csv(submission_path, index=False)
print('Saved submission to', submission_path)
submission.head()

Saved submission to ../submission/submission_NN.csv


,ID,label
0,0,1
1,2,1
2,4,0
3,8,1
4,9,0
